# Exam Score Prediction

**Key Improvements Explained:**

Feature Engineering: I added study_efficiency and total_engagement. In "Playground" datasets, these interactions often help the model distinguish between a student who studies hard but misses class versus one who does both.

OrdinalEncoder: For tree-based models, OrdinalEncoder is generally superior to OneHotEncoder. It preserves the categorical information in a single column, allowing trees to make more meaningful splits without creating high-dimensional, sparse data.

Weighted Blending: Instead of choosing between XGBoost and LightGBM, we use both. Since they "learn" the data differently, their errors often cancel each other out, leading to a lower RMSE on the leaderboard.

In [ ]:
# Configuration and Libraries Loading

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import optuna
import logging
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Suppress logs for a cleaner output
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

## Load Data and Analyze

In [ ]:
# data
path        = '/kaggle/input/playground-series-s6e1/'
df_train = pd.read_csv(path + 'train.csv',             index_col = 'id')
df_test = pd.read_csv(path + 'test.csv',              index_col = 'id')
submission  = pd.read_csv(path + 'sample_submission.csv', index_col = 'id')


## Data QC and EDA

In [ ]:
# Quick data overview
print("\n=== DATA OVERVIEW ===")
print(f"\nTrain columns: {list(df_train.columns)}")
print(f"\nTest columns: {list(df_test.columns)}")
print(f"\nData types:\n{df_train.dtypes.value_counts()}")

# Check for missing values
print(f"\nMissing values in train:\n{df_train.isnull().sum().sum()} total")
print(f"Missing values in test:\n{df_test.isnull().sum().sum()} total")

# Define Target
TARGET = "exam_score"

# Target distribution
print(f"\nData Statistics:")
print(df_train.describe().T)


In [ ]:
print("=== EXPLORATORY DATA ANALYSIS ===\n")

# Target Distribution Analysis
plt.figure(figsize=(12, 5))

# Histogram
plt.subplot(1, 2, 1)
plt.hist(
    df_train["exam_score"],
    bins=50,
    alpha=0.7,
    edgecolor='black'
)
plt.title('Target Distribution')
plt.xlabel('Exam Score')
plt.ylabel('Frequency')

# Boxplot
plt.subplot(1, 2, 2)
plt.boxplot(df_train["exam_score"], vert=True)
plt.title('Target Box Plot')
plt.ylabel('Exam Score')

plt.tight_layout()
plt.show()


In [ ]:
# Feature Type Identification
categorical_cols = df_train.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
numerical_cols.remove(TARGET)  # Remove target

# Correlation Analysis
plt.figure(figsize=(15, 12))
correlation_matrix = df_train[numerical_cols + [TARGET]].corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# Show top correlations with target
print("\nTop correlations with target:")
target_corr = correlation_matrix[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print(target_corr.head(10))

## Feature Engineering

In [ ]:
# Feature Engineering Function
def engineer_features(df):
    df = df.copy()
    # Efficiency Score: ratio of study to sleep
    df['study_efficiency'] = df['study_hours'] / (df['sleep_hours'] + 1)
    # Total Engagement: product of attendance and study hours
    df['total_engagement'] = (df['class_attendance'] / 100) * df['study_hours']
    return df

X = engineer_features(df_train.drop(columns=["exam_score"]))
y = df_train["exam_score"]
X_test = engineer_features(df_test)

## Preprocessing 

In [ ]:
# Updated Preprocessing
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols)
    ]
)

## Optimization of ML Algorithms

In [ ]:
def objective_lgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 128),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 2.0),
        "verbosity": -1,
        "random_state": RANDOM_STATE
    }

    model = LGBMRegressor(**params)

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    scores = cross_val_score(
        pipe,
        X,
        y,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    return -scores.mean()


## Run LGBM Optimization

In [ ]:
print("Optimizing LGBMRegressor...")
study_lgbm = optuna.create_study(direction="minimize")
study_lgbm.optimize(objective_lgbm, n_trials=6)

print(f"Best LGBM RMSE: {study_lgbm.best_value:.4f}")

best_lgbm = LGBMRegressor(
    **study_lgbm.best_params,
    random_state=RANDOM_STATE
)


In [ ]:
def objective_xgboost(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 2000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }

    model = XGBRegressor(**params)

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    scores = cross_val_score(
        pipe,
        X,
        y,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    return -scores.mean()


In [ ]:
print("Optimizing XGBoost...")
study_xgb = optuna.create_study(direction="minimize")
study_xgb.optimize(objective_xgboost, n_trials=30)

print(f"Best XGBoost RMSE: {study_xgb.best_value:.4f}")

best_xgb = XGBRegressor(
    **study_xgb.best_params,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1
)


## Fitting dataset with best parameters for LGBM and XGB

In [ ]:
pipe_lgbm = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_lgbm)
]).fit(X, y)

pipe_xgb = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_xgb)
]).fit(X, y)


## Apply Model on test dataset and apply weighted blending

In [ ]:
lgbm_preds = pipe_lgbm.predict(X_test)
xgb_preds = pipe_xgb.predict(X_test)

final_predictions = 0.4 * lgbm_preds + 0.6 * xgb_preds


## Save Data and Submit

In [ ]:
submission  = pd.read_csv(path + 'sample_submission.csv', index_col = 'id')

submission['exam_score'] = final_predictions
submission.to_csv('submission.csv')
submission.head(10)